# Run calibration

Outcomes:
- Prepare ACS demographic targets and ATUS conditional cluster distribution table
- Assign mobility users to ATUS behavioral clusters via k-NN on the precomputed distance matrix
- Run two-stage calibration (IPF demographic + behavioral raking) with replicate weights
- Export main and replicate weights

**Pipeline position:** follows `walkthrough_02_process_acs.ipynb` and `walkthrough_03_process_atus.ipynb`

In [ ]:
import numpy as np
import pandas as pd

%load_ext autoreload
%autoreload 2
from helpers import prep_calibration_inputs

from mobcalibrate import Calibrator

In [ ]:
# ======================
# PATHS
# ======================
DATA_DIR = "data/processed"
CBSA_CODE = 38060

# Inputs from previous steps
DIST_MATRIX_FILE = f"{DATA_DIR}/distance_matrix_3cat_full_embedding.parquet"
ATUS_META_FILE   = f"{DATA_DIR}/atus_meta_{CBSA_CODE}.csv"
ACS_CBG_FILE     = f"{DATA_DIR}/acs_cbg_distr_{CBSA_CODE}.csv"
INCOME_MARGIN_FILE = f"{DATA_DIR}/acs_income_margins_{CBSA_CODE}.csv"
AGE_MARGIN_FILE    = f"{DATA_DIR}/acs_age_margins_{CBSA_CODE}.csv"
MEDOID_INFO_FILE = f"{DATA_DIR}/atus_medoid_info_{CBSA_CODE}.json"

# Outputs
OUT_CALIBRATION_RESULTS = f"{DATA_DIR}/calibration_results_{CBSA_CODE}.pkl"
OUT_WEIGHTS = f"{DATA_DIR}/weights_{CBSA_CODE}.parquet"

# ======================
# STRATIFICATION VARIABLES
# (must match the coding used in walkthrough_03)
# ======================
ROW_VAR          = "income"       # row stratification variable (matches atus_meta column)
COL_VAR          = "age"          # col stratification variable (matches atus_meta column)
CLUSTER_LABEL_COL = "cluster_label"
WEIGHT_COL        = "TUFINLWGT"
GEOID_COL         = "GEOID"

NUM_CLUSTERS = 4

# ======================
# KNN ASSIGNMENT
# ======================
KNN_K          = 10    # number of nearest ATUS neighbors
KNN_THRESHOLD  = 0.5   # min fraction of neighbors required to agree on a label

# ======================
# CALIBRATOR SETTINGS
# ======================
NUM_REPLICATES = 51
SEED           = 1234

## 1. Load data

In [ ]:
atus_meta  = pd.read_csv(ATUS_META_FILE)
dist       = pd.read_parquet(DIST_MATRIX_FILE)
acs_cbg    = pd.read_csv(ACS_CBG_FILE, dtype={GEOID_COL: str})
row_margin = pd.read_csv(INCOME_MARGIN_FILE)
col_margin = pd.read_csv(AGE_MARGIN_FILE)
medoid_info = prep_calibration_inputs.load_medoid_info(MEDOID_INFO_FILE)

print(f"ATUS respondents: {len(atus_meta)}")
print(f"Mobility users:   {len(dist)}")
print(f"CBGs in ACS:      {len(acs_cbg)}")

In [ ]:
# hypothetical user ids (since real ones cannot be published due to data agreement)
dist['user_id'] = range(1, len(dist)+1)

# fix bug in distance matrix
dist['20190504191857'] = dist.loc[:, '20190504191857'].str.replace('\x18', '').astype(float)

## 2. Prepare ACS targets

`prep_acs_targets` normalizes the marginal counts into probability distributions and extracts category label arrays (used to size the Calibrator and label outputs).

By default the Calibrator runs stage-1 **IPF** against the two independent marginals (`acs_row_margin`, `acs_col_margin`). If a CBSA-level *joint* target is available (e.g. from PUMS microdata), pass it as `acs_joint_margin` instead to trigger single-margin raking against the joint — strictly stronger than two-marginal IPF. The next cell shows the required matrix shape using an illustrative joint built from the marginals' outer product.

In [ ]:
acs_targets = prep_calibration_inputs.prep_acs_targets(
    row_margin_df = row_margin,
    col_margin_df = col_margin,
    row_var = "hh_income",
    col_var = "age_group",
)

print("Row categories (income):", acs_targets["row_cats"])
print("Col categories (age):   ", acs_targets["col_cats"])
print("Target population:      ", acs_targets["target_pop_tot"])

# (Optional) joint target for stage-1 raking. The Calibrator expects the joint
# in the SAME row/col category order as acs_row_cats/acs_col_cats; stage1_rake
# flattens it row-major (income outer, age inner). With a real joint (e.g. PUMS)
# you'd pivot/reindex it to this shape. Here we use the independence joint
# (outer product of the marginals) purely to illustrate the required shape.
joint_matrix = np.outer(acs_targets["row_margin"], acs_targets["col_margin"])
print(f"Joint target shape: {joint_matrix.shape}  (income * age),  sum = {joint_matrix.sum():.6f}")

## 3. Prepare ATUS behavioral target table

Computes P(cluster | demographic stratum) from ATUS respondent weights.
Rows index joint demographic strata (income × age), columns index clusters.
Each row sums to 1.

In [ ]:
atus_target_P = prep_calibration_inputs.prep_atus_target(
    atus_meta_df      = atus_meta,
    row_var           = ROW_VAR,
    col_var           = COL_VAR,
    cluster_label_col = CLUSTER_LABEL_COL,
    weight_col        = WEIGHT_COL,
    num_row_cats      = acs_targets["num_row_cats"],
    num_col_cats      = acs_targets["num_col_cats"],
    num_clusters      = NUM_CLUSTERS,
)

print(f"Target table shape: {atus_target_P.shape}  (strata * clusters)")
atus_target_P

## 4. Assign mobility users to ATUS clusters

Uses k-NN voting on the precomputed distance matrix. Each mobility user is
assigned the plurality cluster among their `KNN_K` nearest ATUS neighbors,
provided that cluster accounts for at least `KNN_THRESHOLD` of those neighbors.
Users below the threshold, or farther than the medoid distance threshold for
their cluster, are marked unassigned (`-1`) and excluded from the behavioral
raking step (their demographic weights are still computed in stage 1).

In [ ]:
assigned_labels = prep_calibration_inputs.assign_mobility_clusters(
    dist_df           = dist,
    atus_meta_df      = atus_meta,
    cluster_label_col = CLUSTER_LABEL_COL,
    k                 = KNN_K,
    threshold         = KNN_THRESHOLD,
    medoid_indices    = medoid_info['medoid_indices'],
    medoid_thresholds = medoid_info['medoid_thresholds'],
)

n_assigned   = int((assigned_labels >= 0).sum())
n_unassigned = int((assigned_labels == -1).sum())
print(f"Assigned:   {n_assigned}  ({n_assigned / len(assigned_labels):.1%})")
print(f"Unassigned: {n_unassigned}  ({n_unassigned / len(assigned_labels):.1%})")
#pd.Series(assigned_labels).value_counts().sort_index().rename("count")

## 5. Filter to users with valid home CBGs

Drops mobility users whose home GEOID does not appear in the ACS CBG table.
These users cannot be calibrated because no demographic distribution is available
for their home census block group.

In [ ]:
user_ids, home_cbgs, assigned_labels, n_dropped = prep_calibration_inputs.filter_valid_users(
    users_df        = dist,
    acs_cbg_df      = acs_cbg,
    assigned_labels = assigned_labels,
    geoid_col       = GEOID_COL,
)

print(f"Users retained: {len(home_cbgs)}")
print(f"Users dropped (missing CBG): {n_dropped}")
print("Proportion of assigned cluster labels:")
pd.Series(assigned_labels).value_counts(normalize=True).sort_index().rename("proportion")

In [ ]:
print(user_ids[:5])
print(home_cbgs[:5])
print(assigned_labels[:5])

## 6. Run calibration

Initializes the `Calibrator` and runs the two-stage procedure for the main weight
set and all replicates. Each replicate independently samples demographic codes from
the CBG-level ACS distributions before raking, propagating individual-level
demographic uncertainty into the weight variance.

`create_main_weights()` and `create_replicate_weights()` both use the same `mode`
argument, which controls which calibration stages are run. The default `"behavioural_full"`
runs both stage 1 (demographic IPF) and stage 2 (behavioral raking).

In [ ]:
calibrator = Calibrator(
    unit_ids               = user_ids,
    home_cbgs              = home_cbgs,
    assigned_cluster_labels = assigned_labels,
    acs_cbg_probs_df       = acs_cbg,
    acs_row_var_name       = ROW_VAR,
    acs_col_var_name       = COL_VAR,
    acs_row_cats           = acs_targets["row_cats"],
    acs_col_cats           = acs_targets["col_cats"],
    #acs_joint_margin       = joint_matrix,   # optional: joint-rake stage 1 (supply a real joint target)
    # Default: independent marginals -> two-margin IPF stage 1:
    acs_row_margin = acs_targets["row_margin"],
    acs_col_margin = acs_targets["col_margin"],
    atus_target_table      = atus_target_P,
    target_pop_tot         = acs_targets["target_pop_tot"],
    geoid_col              = GEOID_COL,
    num_replicates         = NUM_REPLICATES,
    seed                   = SEED,
)

In [ ]:
CALIBRATION_MODE = "demographic_behavioral"
calibration_results = calibrator.create_weights(mode = CALIBRATION_MODE)

## 7. Inspect weights

In [ ]:
calibration_results.all_final_weights()

In [ ]:
calibration_results.to_df(replicate_id=0) # replicate_id=0 => main weights

In [ ]:
calibration_results.to_long_df()

In [ ]:
# Stage-1 calibrated joint demographic distribution of the sample.
#
# Stage 1 runs inside Calibrator._compute_one_replicate during create_weights():
#   * stage1_ipf   if acs_row_margin + acs_col_margin were supplied  (the path used here)
#   * stage1_rake  if acs_joint_margin was supplied instead
# Both return `weight1` — the stage-1 weight per sample unit. Crosstabbing
# weight1 by the sampled (income, age) stratum codes gives the calibrated
# joint demographic distribution of the sample.

df_main  = calibration_results.to_df(replicate_id=0)   # main weights
row_cats = acs_targets["row_cats"]                     # income labels (order matches codes)
col_cats = acs_targets["col_cats"]                     # age labels (order matches codes)

sample_joint_p = pd.crosstab(
    pd.Categorical.from_codes(df_main[f"sampled_{ROW_VAR}_code"], categories=row_cats, ordered=True),
    pd.Categorical.from_codes(df_main[f"sampled_{COL_VAR}_code"], categories=col_cats, ordered=True),
    values=df_main["weight1"], aggfunc="sum",
    rownames=[ROW_VAR], colnames=[COL_VAR], dropna=False, normalize=True
)
sample_joint_p

## 8. Export

In [ ]:
# save entire object as pickle
calibration_results.save(OUT_CALIBRATION_RESULTS)
print(f"CalibrationResult object saved to: {OUT_CALIBRATION_RESULTS}")

# or save weights df directly as either csv or parquet
calibration_results.to_long_df().to_parquet(OUT_WEIGHTS)
print(f"Weights df saved to: {OUT_WEIGHTS}")


# load back results:
#calibration_results = CalibrationResult.load(OUT_CALIBRATION_RESULTS)